## Databricks - HR Analytics/Data Transformations



In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import (
    col, when, lit, regexp_replace, initcap, lower, upper,
    current_date, datediff, to_date, year, month, dayofmonth,
    array, explode, collect_list, count, sum as spark_sum, udf
)
from pyspark.sql.types import StringType, IntegerType
from pyspark.sql.window import Window


## Data Reading


In [0]:
input_path = "/Workspace/Users/lavanus98@gmail.com/HR-Analytics-Databricks-Transformations/Source.csv"

df = spark.read.csv(
    input_path,
    header=True,
    inferSchema=True
)

display(df)


In [0]:
df.printSchema()


In [0]:
df.show()


## 1. Column Name Rename


In [0]:
df = df.withColumnRenamed("gender", "employee_gender") \
       .withColumnRenamed("company_size", "employee_count")

display(df)


## 2. When - Otherwise, Col, Lit


In [0]:
df = df.withColumn(
    "relevent_experience",
    when(col("relevent_experience") == "Has relevent experience", lit(1)).otherwise(lit(0))
)

df.display()


## 3. withColumn, regexp_replace, col


In [0]:
df = df.withColumn(
    "experience",
    when(col("experience").isNotNull(), regexp_replace(col("experience"), "[^0-9]", ""))
    .otherwise(None)
).withColumn(
    "last_new_job_clean",
    when(col("last_new_job").isNotNull(), regexp_replace(col("last_new_job"), "[^0-9]", ""))
    .otherwise(None)
)

display(df)


## 4. FillNa


In [0]:
df = df.fillna({
    "employee_gender": "Unknown",
    "major_discipline": "Unknown",
    "company_type": "Unknown",
    "employee_count": "Unknown"
})

display(df)


## 5. Select, Alias


In [0]:
selected_df = df.select(
    col("enrollee_id").alias("employee_id"),
    col("city"),
    col("city_development_index").alias("dev_index"),
    col("employee_gender"),
    col("relevent_experience"),
    col("education_level"),
    col("major_discipline"),
    col("experience"),
    col("employee_count"),
    col("company_type"),
    col("training_hours"),
    col("target")
)

display(selected_df)


## 6. Filter


In [0]:
filtered_df = selected_df.filter(
    (col("training_hours") > 50) & (col("relevent_experience") == 1)
)

display(filtered_df)


## 7. Sort


In [0]:
sorted_df = filtered_df.orderBy(col("training_hours").desc(), col("dev_index").desc())

display(sorted_df)


## 8. DropDuplicates


In [0]:
dedup_df = selected_df.dropDuplicates(["employee_id"])

display(dedup_df)


## 9. Select, Initcap, Lower, Upper


In [0]:
text_df = df.select(
    col("enrollee_id"),
    initcap(col("major_discipline")).alias("major_discipline_initcap"),
    lower(col("company_type")).alias("company_type_lower"),
    upper(col("employee_gender")).alias("employee_gender_upper")
)

display(text_df)


## 10. DropNa


In [0]:
dropna_df = df.dropna(subset=["education_level", "major_discipline"])

display(dropna_df)


## 11. FillNa - Another Example


In [0]:
fillna_df = df.fillna({
    "education_level": "Not Provided",
    "enrolled_university": "Not Provided",
    "experience": "0",
    "last_new_job_clean": "0"
})

display(fillna_df)


## 12. Drop Column


In [0]:
dropcol_df = fillna_df.drop("relevent_experience")

display(dropcol_df)


## 13. Joins - Inner, Left, Right, Outer


In [0]:
city_lookup = spark.createDataFrame(
    [
        ("city_103", "Tier 1"),
        ("city_21", "Tier 1"),
        ("city_160", "Tier 2"),
        ("city_114", "Tier 2"),
        ("city_173", "Tier 3")
    ],
    ["city", "city_tier"]
)

inner_join_df = df.join(city_lookup, on="city", how="inner")
left_join_df = df.join(city_lookup, on="city", how="left")
right_join_df = df.join(city_lookup, on="city", how="right")
outer_join_df = df.join(city_lookup, on="city", how="outer")

display(inner_join_df)


## 14. Union & UnionByName


In [0]:
part_a = selected_df.limit(10)
part_b = selected_df.filter(col("employee_gender") == "Male").limit(10)

union_df = part_a.union(part_b)

part_c = part_a.select(
    "employee_id", "city", "dev_index", "employee_gender", "relevent_experience",
     "education_level", "major_discipline",
    "experience", "employee_count", "company_type",
    "training_hours", "target"
)

part_d = part_b.select(
    "city", "employee_id", "employee_gender", "dev_index", "relevent_experience",
 "education_level", "major_discipline",
    "experience", "employee_count", "company_type",
    "training_hours", "target"
)

union_by_name_df = part_c.unionByName(part_d, allowMissingColumns=True)

display(union_df)


## 15. Date Functions


In [0]:
date_df = df.withColumn("load_date", current_date()) \
            .withColumn("sample_join_date", to_date(lit("2024-01-15"))) \
            .withColumn("days_since_join", datediff(col("load_date"), col("sample_join_date"))) \
            .withColumn("join_year", year(col("sample_join_date"))) \
            .withColumn("join_month", month(col("sample_join_date"))) \
            .withColumn("join_day", dayofmonth(col("sample_join_date")))

display(date_df)


## 16. Array


In [0]:
array_df = df.withColumn(
    "profile_tags",
    array(col("education_level"), col("major_discipline"), col("company_type"))
)

display(array_df)


## 17. Explode


In [0]:
explode_df = array_df.select("enrollee_id", explode(col("profile_tags")).alias("tag"))

display(explode_df)


## 18. Collect_List


In [0]:
collect_df = df.groupBy("city").agg(
    collect_list("education_level").alias("education_levels")
)

display(collect_df)


## 19. Count


In [0]:
count_df = df.groupBy("city").agg(
    count("*").alias("candidate_count")
).orderBy(col("candidate_count").desc())

display(count_df)


## 20. Pivot


In [0]:
pivot_df = df.groupBy("education_level").pivot("employee_gender").agg(count(" "))

display(pivot_df)


## 21. When-Otherwise


In [0]:
status_df = df.withColumn(
    "training_bucket",
    when(col("training_hours") >= 100, "High")
    .when(col("training_hours") >= 50, "Medium")
    .otherwise("Low")
)

display(status_df)


## 22. Window - Rank & Dense Rank


In [0]:
w = Window.partitionBy("city").orderBy(col("training_hours").desc())

window_df = df.withColumn("rank_in_city", F.rank().over(w)) \
              .withColumn("dense_rank_in_city", F.dense_rank().over(w))

display(window_df)


## 23. Cumulative Sum


In [0]:
cum_window = Window.partitionBy("city").orderBy(col("training_hours")) \
                   .rowsBetween(Window.unboundedPreceding, Window.currentRow)

cumulative_df = df.withColumn(
    "cumulative_training_hours",
    spark_sum("training_hours").over(cum_window)
)

display(cumulative_df)


## 24. User Defined Function


In [0]:
def exp_bucket(val):
    if val is None or str(val).strip() == "":
        return "Unknown"
    try:
        x = int(val)
        if x < 3:
            return "Beginner"
        elif x < 8:
            return "Intermediate"
        else:
            return "Experienced"
    except:
        return "Unknown"

exp_bucket_udf = udf(exp_bucket, StringType())

udf_df = df.withColumn("experience_bucket", exp_bucket_udf(col("experience_clean")))

display(udf_df)


## 25. Data Export with different modes & different formats


In [0]:
final_df = udf_df

final_df.write.mode("overwrite").csv("/FileStore/tables/fullcourse/fullcsv.csv")
final_df.write.format("delta").mode("append").save("/FileStore/tables/fulldelta/deltasave")
final_df.write.mode("overwrite").parquet("/FileStore/tables/fullcourse/fullparquet")

display(final_df)
